In [2]:
# Library Import
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Importing Dataset
file_path = "../data/Loan Default Data.xlsx"
df = pd.read_excel(file_path)

# Display df
print(df.shape)
print(df.info())
print(df.head())

(148670, 34)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148670 entries, 0 to 148669
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   ID                         148670 non-null  int64  
 1   year                       148670 non-null  int64  
 2   loan_limit                 145326 non-null  object 
 3   Gender                     148670 non-null  object 
 4   approv_in_adv              147762 non-null  object 
 5   loan_type                  148670 non-null  object 
 6   loan_purpose               148536 non-null  object 
 7   Credit_Worthiness          148670 non-null  object 
 8   open_credit                148670 non-null  object 
 9   business_or_commercial     148670 non-null  object 
 10  loan_amount                148670 non-null  int64  
 11  rate_of_interest           112231 non-null  float64
 12  Interest_rate_spread       112031 non-null  float64
 13  Upfront_charges 

In [3]:
# Columns to keep based on the Research Question
keep_cols = [
    # Demographics and Loan Categories
    "Gender", "loan_limit", "loan_purpose", "business_or_commercial", 
    "Region", "age", "credit_type", "occupancy_type",
    
    # Financial/Quantitative metrics
    "loan_amount", "income", "Credit_Score", "dtir1", "property_value",
    
    # Target variable (1 = Default, 0 = No Default)
    "Status"
]

# Keep only the selected columns
df = df[keep_cols]

# Quick check
print(df.info())
print(f"Dataset now has {df.shape[0]} rows and {df.shape[1]} columns")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148670 entries, 0 to 148669
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Gender                  148670 non-null  object 
 1   loan_limit              145326 non-null  object 
 2   loan_purpose            148536 non-null  object 
 3   business_or_commercial  148670 non-null  object 
 4   Region                  148670 non-null  object 
 5   age                     148470 non-null  object 
 6   credit_type             148670 non-null  object 
 7   occupancy_type          148670 non-null  object 
 8   loan_amount             148670 non-null  int64  
 9   income                  139520 non-null  float64
 10  Credit_Score            148670 non-null  int64  
 11  dtir1                   124549 non-null  float64
 12  property_value          133572 non-null  float64
 13  Status                  148670 non-null  int64  
dtypes: float64(3), int64

In [4]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Handle missing values by dropping them to preserve clean data without imputation bias
df = df.dropna() 
print(f"Dataset shape after handling missing values: {df.shape}")

Missing values per column:
Gender                        0
loan_limit                 3344
loan_purpose                134
business_or_commercial        0
Region                        0
age                         200
credit_type                   0
occupancy_type                0
loan_amount                   0
income                     9150
Credit_Score                  0
dtir1                     24121
property_value            15098
Status                        0
dtype: int64
Dataset shape after handling missing values: (121447, 14)


In [5]:
# Loop through and print unique text variables
for column in df.columns:
    if df[column].dtype == 'object': 
        print(f"Unique values in column '{column}':")
        print(df[column].unique())
        print()

Unique values in column 'Gender':
['Sex Not Available' 'Male' 'Joint' 'Female']

Unique values in column 'loan_limit':
['cf' 'ncf']

Unique values in column 'loan_purpose':
['p1' 'p4' 'p3' 'p2']

Unique values in column 'business_or_commercial':
['nob/c' 'b/c']

Unique values in column 'Region':
['south' 'North' 'central' 'North-East']

Unique values in column 'age':
['25-34' '35-44' '45-54' '55-64' '65-74' '>74' '<25']

Unique values in column 'credit_type':
['EXP' 'CRIF' 'CIB' 'EQUI']

Unique values in column 'occupancy_type':
['pr' 'sr' 'ir']



In [6]:
# Start from your selected columns
df_encoded = df.copy()

# Identify text columns to One-Hot Encode
cat_cols = [
    "Gender", "loan_limit", "loan_purpose", "business_or_commercial", 
    "Region", "age", "credit_type", "occupancy_type"
]

# One-hot encode multi-category variables
df_encoded = pd.get_dummies(
    df_encoded,
    columns=cat_cols,
    drop_first=False
)

# Convert all boolean columns to integers (0/1)
bool_cols = df_encoded.select_dtypes(include=['bool']).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

# Target 'Status' distribution check
print("Target Variable Distribution:")
print(df_encoded['Status'].value_counts())

Target Variable Distribution:
Status
0    101631
1     19816
Name: count, dtype: int64


In [7]:
# Save the cleaned dataset
df_encoded.to_csv('loan_cleaned_encoded.csv', index=False)
print("Cleaned dataset saved as 'loan_cleaned_encoded.csv'")
print(f"Dataset shape: {df_encoded.shape}")

Cleaned dataset saved as 'loan_cleaned_encoded.csv'
Dataset shape: (121447, 36)


In [8]:
# Separate features (X) and target (y)
X = df_encoded.drop('Status', axis=1)
y = df_encoded['Status']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features shape: (121447, 35)
Target distribution:
Status
0    101631
1     19816
Name: count, dtype: int64


In [9]:
# Split into train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Split temp into validation (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

Train: (85012, 35) Validation: (18217, 35) Test: (18218, 35)


In [10]:
# Save features and labels for each split
X_train.to_csv("X_train.csv", index=False)
y_train.to_csv("y_train.csv", index=False)

X_val.to_csv("X_val.csv", index=False)
y_val.to_csv("y_val.csv", index=False)

X_test.to_csv("X_test.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Datasets saved as CSV files:")
print("X_train.csv, y_train.csv, X_val.csv, y_val.csv, X_test.csv, y_test.csv")

Datasets saved as CSV files:
X_train.csv, y_train.csv, X_val.csv, y_val.csv, X_test.csv, y_test.csv


In [11]:
# Train base model
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predict on validation set
y_val_pred = rf.predict(X_val)
y_val_prob = rf.predict_proba(X_val)[:, 1]

# Metrics
print("Accuracy :", round(accuracy_score(y_val, y_val_pred), 3))
print("Precision:", round(precision_score(y_val, y_val_pred), 3))
print("Recall   :", round(recall_score(y_val, y_val_pred), 3))
print("F1-score :", round(f1_score(y_val, y_val_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_prob), 3))

print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_val_pred))
print("\nClassification Report:\n", classification_report(y_val, y_val_pred, digits=3))

Accuracy : 0.853
Precision: 0.758
Recall   : 0.148
F1-score : 0.248
ROC-AUC  : 0.733

Confusion Matrix:
 [[15104   141]
 [ 2531   441]]

Classification Report:
               precision    recall  f1-score   support

           0      0.856     0.991     0.919     15245
           1      0.758     0.148     0.248      2972

    accuracy                          0.853     18217
   macro avg      0.807     0.570     0.583     18217
weighted avg      0.840     0.853     0.809     18217



In [12]:
from sklearn.model_selection import GridSearchCV

# Define base model
rf_grid = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

# GridSearch with 5-fold CV
grid_search = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score (ROC-AUC):", grid_search.best_score_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
Best CV Score (ROC-AUC): 0.740178309168502


In [13]:
# Final tuned model using the best parameters from GridSearchCV
best_rf = RandomForestClassifier(
    n_estimators=500,       
    max_depth=None,         
    min_samples_split=2,    
    min_samples_leaf=2,     
    max_features='sqrt',    
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train the model on the training data
best_rf.fit(X_train, y_train)

# Predict on the unseen testing set
y_test_pred = best_rf.predict(X_test)
y_test_prob = best_rf.predict_proba(X_test)[:, 1]

# Print out the final performance metrics
print("Accuracy :", round(accuracy_score(y_test, y_test_pred), 3))
print("Precision:", round(precision_score(y_test, y_test_pred), 3))
print("Recall   :", round(recall_score(y_test, y_test_pred), 3))
print("F1-score :", round(f1_score(y_test, y_test_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_test_prob), 3))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test, y_test_pred, digits=3))

Accuracy : 0.847
Precision: 0.57
Recall   : 0.265
F1-score : 0.362
ROC-AUC  : 0.737

Confusion Matrix:
 [[14652   593]
 [ 2186   787]]

Classification Report:
               precision    recall  f1-score   support

           0      0.870     0.961     0.913     15245
           1      0.570     0.265     0.362      2973

    accuracy                          0.847     18218
   macro avg      0.720     0.613     0.637     18218
weighted avg      0.821     0.847     0.823     18218

